In [ ]:
#r "..\src\functions\bin\Debug\net9.0\functions.dll"
#r "nuget: Microsoft.Extensions.Logging, 9.0.0"
//#!import "../src/functions/Azure/ContentUnderstanding/AzureContentUnderstandingClient.cs"

using Microsoft.Extensions.Logging;
using Billy.Function.AzureContentUnderstanding;
using Billy.Function.Models;
using Billy.Function.Models.ACM;
using Billy.Function.Parsing;
using System.Text.Json;

var endpoint = "https://billyaiservices783743165220.cognitiveservices.azure.com";
var apiVersion = "2024-12-01-preview";
var subscriptionKey = Environment.GetEnvironmentVariable("ACU_SUBSCRIPTION_KEY");
var apiToken = Environment.GetEnvironmentVariable("ACU_API_TOKEN") ?? "";


var loggerFactory = LoggerFactory.Create(builder =>
{
    builder.AddConsole(options =>
    {
        options.FormatterName = "Simple";
    }).SetMinimumLevel(LogLevel.Information);
    builder.AddSimpleConsole(options =>
    {
        options.SingleLine = true;
        options.IncludeScopes = false;
        options.UseUtcTimestamp = false;
        options.TimestampFormat = "yyyy-MM-dd HH:mm:ss ";
    });
    builder.Configure(options =>
    {
        options.ActivityTrackingOptions = ActivityTrackingOptions.None;
    });
});
var _logger = loggerFactory.CreateLogger<AzureContentUnderstandingClient>();

var client = new AzureContentUnderstandingClient(_logger, endpoint, apiVersion, subscriptionKey, apiToken);
var responseMessage = await client.BeginAnalyzeAsync("BillAnalyzer", "C:\\Users\\Desktop\\OneDrive\\Imágenes\\DELAPAZ.jpg");
_logger.LogInformation("Response message: {responseMessage}", responseMessage.ToString());
var result = await client.PollResultAsync(responseMessage);
// JsonSerializer.Serialize(result, new JsonSerializerOptions { WriteIndented = true })
var json = client.GetJsonFields(result);
Invoice invoice  = InvoiceParser.Parse(json);
_logger.LogInformation($"Invoice Details:");
_logger.LogInformation($"Customer: {invoice.CustomerName}");
_logger.LogInformation($"Amount Due: {invoice.AmountDue}");
_logger.LogInformation($"Invoice Date: {invoice.InvoiceDate:yyyy-MM-dd}");
_logger.LogInformation($"Due Date: {invoice.DueDate:yyyy-MM-dd}");
_logger.LogInformation($"Total Items: {invoice.Items?.Count ?? 0}");
_logger.LogInformation($"TOTAL: {invoice.InvoiceTotal}");

// Display item details if available
if (invoice.Items != null && invoice.Items.Count > 0)
{
    _logger.LogInformation("\nItem Details:");
    foreach (var item in invoice.Items)
    {
        _logger.LogInformation($"- {item.Description}: {item.TotalPrice}");
    }
}

"FINISHED"



Installed Packages Microsoft.Extensions.Logging, 9.0.0

info: Billy.Function.AzureContentUnderstanding.AzureContentUnderstandingClient[0] Analyzing file C:\Users\Desktop\OneDrive\Imágenes\DELAPAZ.jpg with analyzer: BillAnalyzer
info: Billy.Function.AzureContentUnderstanding.AzureContentUnderstandingClient[0] Response message: StatusCode: 202, ReasonPhrase: 'Accepted', Version: 1.1, Content: System.Net.Http.HttpConnectionResponseContent, Headers: {   Transfer-Encoding: chunked   request-id: 59ce4bc4-931e-4b91-9018-410e4282e36b   x-ms-request-id: 59ce4bc4-931e-4b91-9018-410e4282e36b   Operation-Location: https://billyaiservices783743165220.cognitiveservices.azure.com/contentunderstanding/analyzers/BillAnalyzer/results/59ce4bc4-931e-4b91-9018-410e4282e36b?api-version=2024-12-01-preview   api-supported-versions: 2024-12-01-preview,2025-03-31-preview   x-envoy-upstream-service-time: 340   apim-request-id: 59ce4bc4-931e-4b91-9018-410e4282e36b   Strict-Transport-Security: max-age=31536000; includeSubDomains; preload   X-Content-Type-Options: nosni

FINISHED